**PROMPT**
Write a Python script that scrapes the latest article titles from a news or blog website using requests and BeautifulSoup. The script should:

Send an HTTP GET request to the target URL with a proper User-Agent header to avoid being blocked.
Parse the HTML response with BeautifulSoup and extract the text of each article title (and its link, if available) using the appropriate HTML tag/class for that site (e.g., h2.entry-title, h3.post-title, etc. — inspect the site's HTML to find the correct selector).
Store the extracted titles in a list of dictionaries, each containing at least title and link.
Save the results to a JSON file (titles.json) that includes a timestamp of when the scrape ran and the total count of articles found.
Handle errors gracefully — e.g., failed requests, missing tags, or empty results — with clear error messages instead of crashing.
Include comments explaining each step, and make the target URL and CSS selector easy to change for different websites.

Do not scrape sites that disallow it in robots.txt, and avoid sending excessive requests in a short time.

In [ ]:
   !pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

def scrape_article_titles(url, headers=None):
    """
    Scrape article titles from a blog/news website and save to JSON.
    """
    if headers is None:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            )
        }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    titles = []

    # Adjust the tag/class below to match the target site's HTML structure.
    # Common patterns: <h2 class="entry-title">, <h3 class="post-title">, <a class="title">
    for tag in soup.find_all(["h1", "h2", "h3"], class_=lambda c: c and "title" in c.lower()):
        text = tag.get_text(strip=True)
        link_tag = tag.find("a")
        link = link_tag["href"] if link_tag and link_tag.has_attr("href") else None

        if text:
            titles.append({
                "title": text,
                "link": link
            })

    return titles


def save_to_json(data, filename="titles.json"):
    output = {
        "scraped_at": datetime.now().isoformat(),
        "count": len(data),
        "articles": data
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4, ensure_ascii=False)
    print(f"Saved {len(data)} titles to {filename}")


if __name__ == "__main__":
    url = "https://www.exampleblog.com"  # replace with the actual target site
    articles = scrape_article_titles(url)

    if articles:
        save_to_json(articles)
    else:
        print("No titles found — check the CSS selector/tag matches the site's HTML structure.")

Saved 13 titles to titles.json
